# ClauseGuard: fine-tune Legal-BERT

1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells and upload `data/clauseguard_in_train.csv` when asked
3. Unzip the downloaded `legalbert.zip` into `backend/models/`

In [ ]:
!pip -q install transformers datasets scikit-learn pandas

In [ ]:
from google.colab import files
files.upload()

## Data and labels

In [ ]:
import json, random, re, time
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

LABELS = ["limitation_of_liability", "unilateral_termination", "unilateral_change", "content_removal",
          "contract_by_using", "choice_of_law", "jurisdiction", "arbitration", "service_bond_lock_in",
          "deposit_forfeiture", "one_sided_notice", "rent_escalation", "non_compete", "broad_data_sharing"]
DATA = Path(".")
MODELS = Path("models")
SEED = 42
BASE = "nlpaueb/legal-bert-small-uncased"
IN_OVERSAMPLE = 3


def _key(name: str) -> str:
    return re.sub(r"[^a-z]+", "_", name.lower()).strip("_")


# load UNFAIR-ToS with our 14 labels
def load_unfair_tos(split: str) -> Tuple[List[str], np.ndarray]:
    from datasets import load_dataset

    ds = load_dataset("coastalcph/lex_glue", "unfair_tos", split=split)
    names = ds.features["labels"].feature.names
    cols = [LABELS.index(_key(n)) for n in names]
    y = np.zeros((len(ds), len(LABELS)), dtype=int)
    for i, labs in enumerate(ds["labels"]):
        for l in labs:
            y[i, cols[l]] = 1
    return list(ds["text"]), y


# load ClauseGuard-IN
def load_in(split: str) -> Tuple[List[str], np.ndarray, pd.DataFrame]:
    name = "clauseguard_in.csv" if split == "all" else f"clauseguard_in_{split}.csv"
    df = pd.read_csv(DATA / name, keep_default_na=False)
    y = np.zeros((len(df), len(LABELS)), dtype=int)
    for i, labs in enumerate(df["labels"]):
        for l in filter(None, str(labs).split("|")):
            y[i, LABELS.index(l)] = 1
    return list(df["text"]), y, df


# pick a small dev set
def group_holdout(df: pd.DataFrame, frac: float = 0.2) -> np.ndarray:
    rng = random.Random(SEED)
    groups = sorted(df["group"].unique())
    rng.shuffle(groups)
    dev = set(groups[: max(1, int(len(groups) * frac))])
    return df["group"].isin(dev).to_numpy()


# best threshold for each label
def tune_thresholds(y_true: np.ndarray, probs: np.ndarray) -> dict:
    from sklearn.metrics import f1_score

    out = {}
    for j, lab in enumerate(LABELS):
        if y_true[:, j].sum() == 0:
            out[lab] = 0.5
            continue
        best_t, best_f = 0.5, -1.0
        for t in np.arange(0.3, 0.91, 0.05):
            f = f1_score(y_true[:, j], (probs[:, j] >= t).astype(int), zero_division=0)
            if f > best_f:
                best_t, best_f = float(round(t, 2)), f
        out[lab] = best_t
    return out

## Training

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def make_loader(texts, y, tok, max_len, batch, shuffle):
    data = list(zip(texts, y.astype("float32")))

    def collate(items):
        enc = tok([t for t, _ in items], truncation=True, max_length=max_len, padding=True, return_tensors="pt")
        enc["labels"] = torch.tensor(np.stack([l for _, l in items]))
        return enc

    return DataLoader(data, batch_size=batch, shuffle=shuffle, collate_fn=collate)


@torch.no_grad()
def predict(model, loader, device) -> np.ndarray:
    model.eval()
    out = []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items() if k != "labels"}
        with torch.autocast(device, enabled=device == "cuda"):
            out.append(torch.sigmoid(model(**b).logits.float()).cpu().numpy())
    return np.vstack(out)


def load_data():
    tos_x, tos_y = load_unfair_tos("train")
    tos_vx, tos_vy = load_unfair_tos("validation")
    in_x, in_y, in_df = load_in("train")
    dev = group_holdout(in_df)
    tr_x = tos_x + [x for x, d in zip(in_x, dev) if not d] * IN_OVERSAMPLE
    tr_y = np.vstack([tos_y] + [in_y[~dev]] * IN_OVERSAMPLE)
    dev_x = tos_vx + [x for x, d in zip(in_x, dev) if d]
    dev_y = np.vstack([tos_vy, in_y[dev]])
    return tr_x, tr_y, dev_x, dev_y


def train(epochs=4, max_len=256, batch=16, lr=5e-5) -> None:
    set_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"device={device}")
    tr_x, tr_y, dev_x, dev_y = load_data()

    tok = AutoTokenizer.from_pretrained(BASE)
    model = AutoModelForSequenceClassification.from_pretrained(
        BASE, num_labels=len(LABELS), problem_type="multi_label_classification",
        id2label=dict(enumerate(LABELS)), label2id={l: i for i, l in enumerate(LABELS)},
    ).to(device)

    # give rare labels more weight
    pos = tr_y.sum(axis=0)
    pos_weight = torch.tensor(np.clip(np.sqrt((len(tr_y) - pos) / np.maximum(pos, 1)), 1.0, 15.0), dtype=torch.float32)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

    train_dl = make_loader(tr_x, tr_y, tok, max_len, batch, True)
    dev_dl = make_loader(dev_x, dev_y, tok, max_len, 64, False)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = len(train_dl) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    scaler = torch.amp.GradScaler(enabled=device == "cuda")

    best_f1, best_state = -1.0, None
    for epoch in range(epochs):
        model.train()
        t0, total = time.time(), 0.0
        for b in train_dl:
            b = {k: v.to(device) for k, v in b.items()}
            labels = b.pop("labels")
            with torch.autocast(device, enabled=device == "cuda"):
                loss = loss_fn(model(**b).logits.float(), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            opt.zero_grad()
            total += loss.item()
        f1 = f1_score(dev_y, (predict(model, dev_dl, device) >= 0.5).astype(int), average="micro", zero_division=0)
        print(f"epoch {epoch + 1}: loss {total / len(train_dl):.4f}, dev micro-F1 {f1:.4f} ({time.time() - t0:.0f}s)")
        if f1 > best_f1:
            best_f1, best_state = f1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # keep the best epoch and tune a threshold per label
    model.load_state_dict(best_state)
    thresholds = tune_thresholds(dev_y, predict(model, dev_dl, device))
    out = MODELS / "legalbert"
    out.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(out)
    tok.save_pretrained(out)
    (MODELS / "thresholds.json").write_text(json.dumps(thresholds, indent=2))
    print(f"saved to {out}, best dev micro-F1 {best_f1:.4f}")


train()

## Download the model

In [ ]:
!cd models && zip -qr ../legalbert.zip legalbert thresholds.json
files.download("legalbert.zip")